# FerroAnalytics – Pruebas Fase I

Este notebook documenta y prueba el sistema completo de analítica de inventario para ferretería.
Recorre cada módulo en orden: **modelos → importador → almacenamiento → reportes**.

| Módulo | Responsabilidad |
|---|---|
| `modelos.py` | Clases de datos y formato struct binario |
| `importador.py` | Lectura y validación de archivos CSV |
| `almacenamiento.py` | Lectura/escritura de archivos `.dat` con upsert |
| `reportes.py` | Cálculo de estadísticas (sin I/O) |
| `menu.py` | Interfaz de consola (no se prueba aquí) |


## 0. Configuración del entorno

Ajustamos el path para importar desde `src/` y situamos el directorio de trabajo
en la raíz del proyecto (necesario porque `almacenamiento.py` usa rutas relativas
como `data/binarios/`).


In [1]:
import sys, os, shutil

# Raíz del proyecto (un nivel arriba de notebooks/)
RAIZ = os.path.abspath('..')
os.chdir(RAIZ)
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)

# Limpiamos los binarios anteriores para partir de cero
dir_bin = os.path.join('data', 'binarios')
if os.path.exists(dir_bin):
    shutil.rmtree(dir_bin)
os.makedirs(dir_bin)

print(f'Directorio de trabajo: {os.getcwd()}')
print('Directorio binarios limpio:', os.listdir(dir_bin))


Directorio de trabajo: /Users/samueldiaz/Development/Jupyter/FerroAnalytics
Directorio binarios limpio: []


## 1. Módulo de Modelos (`src/modelos.py`)

Define las tres clases de datos y sus formatos `struct` para almacenamiento binario.
Cada clase puede serializarse a bytes (`a_bytes`) y reconstruirse desde bytes (`desde_bytes`).


### 1.1 Tamaños de registro en disco

El prefijo `<` (little-endian) elimina el padding automático de Python,
garantizando registros de longitud fija predecible.


In [2]:
from src.modelos import (
    ProductoAnalitico, CategoriaAnalitica, MovimientoAnalitico,
    FORMATO_PRODUCTO, FORMATO_CATEGORIA, FORMATO_MOVIMIENTO,
    TAMANO_PRODUCTO, TAMANO_CATEGORIA, TAMANO_MOVIMIENTO,
)

print('Formatos struct y tamaños de registro:')
print(f'  ProductoAnalitico   formato={FORMATO_PRODUCTO!r:<18}  {TAMANO_PRODUCTO} bytes/registro')
print(f'  CategoriaAnalitica  formato={FORMATO_CATEGORIA!r:<18}  {TAMANO_CATEGORIA} bytes/registro')
print(f'  MovimientoAnalitico formato={FORMATO_MOVIMIENTO!r:<18}  {TAMANO_MOVIMIENTO} bytes/registro')


Formatos struct y tamaños de registro:
  ProductoAnalitico   formato='<10s40sifii10s'    76 bytes/registro
  CategoriaAnalitica  formato='<i30s'             34 bytes/registro
  MovimientoAnalitico formato='<i10s1si10s40s'    69 bytes/registro


### 1.2 Serialización y deserialización (round-trip)

Verificamos que un objeto convertido a bytes y luego reconstruido
produce exactamente los mismos valores.


In [3]:
from datetime import date

# ProductoAnalitico
p = ProductoAnalitico('TORN-M8', 'Tornillo M8 x 40mm', 1, 0.18, 900, 300, '2026-06-14')
p2 = ProductoAnalitico.desde_bytes(p.a_bytes())
assert p.codigo == p2.codigo and p.stock_actual == p2.stock_actual
print('ProductoAnalitico   round-trip OK →', p2)

# CategoriaAnalitica
c = CategoriaAnalitica(1, 'Tornillería')
c2 = CategoriaAnalitica.desde_bytes(c.a_bytes())
assert c.id == c2.id and c.nombre == c2.nombre
print('CategoriaAnalitica  round-trip OK →', c2)

# MovimientoAnalitico
m = MovimientoAnalitico(1, 'TORN-M8', 'E', 500, '2026-01-10')
m2 = MovimientoAnalitico.desde_bytes(m.a_bytes())
assert m.id_movimiento == m2.id_movimiento and m.tipo == m2.tipo
print('MovimientoAnalitico round-trip OK →', m2)


ProductoAnalitico   round-trip OK → ProductoAnalitico(codigo='TORN-M8', nombre='Tornillo M8 x 40mm', id_categoria=1, precio_unitario=0.18000000715255737, stock_actual=900, stock_minimo=300, fecha_ultima_actualizacion='2026-06-14')
CategoriaAnalitica  round-trip OK → CategoriaAnalitica(id=1, nombre='Tornillería')
MovimientoAnalitico round-trip OK → MovimientoAnalitico(id_movimiento=1, codigo_producto='TORN-M8', tipo='E', cantidad=500, fecha='2026-01-10', lote_origen='')


## 2. Módulo Importador (`src/importador.py`)

Lee archivos CSV, valida cada fila según las reglas de negocio y devuelve
dos listas: registros aceptados y filas rechazadas con su motivo.
**No escribe en disco**: eso es responsabilidad de `almacenamiento.py`.


### 2.1 Importación de categorías


In [4]:
from src.importador import importar_categorias, importar_productos, importar_movimientos

cats_ok, cats_err = importar_categorias('data/entrada/categorias.csv')

print(f'Categorías aceptadas : {len(cats_ok)}')
print(f'Categorías rechazadas: {len(cats_err)}')
print()
for c in cats_ok:
    print(f'  [{c.id}] {c.nombre}')


Categorías aceptadas : 6
Categorías rechazadas: 0

  [1] Tornillería
  [2] Herramientas Manuales
  [3] Herramientas Eléctricas
  [4] Pintura y Acabados
  [5] Electricidad
  [6] Plomería


### 2.2 Importación de productos


In [5]:
prod_ok, prod_err = importar_productos('data/entrada/productos.csv')

print(f'Productos aceptados : {len(prod_ok)}')
print(f'Productos rechazados: {len(prod_err)}')
print()
print(f'  {"CÓDIGO":<10} {"NOMBRE":<32} {"CAT":>4} {"PRECIO":>8} {"STOCK":>6} {"MÍN":>5}')
print('  ' + '-'*68)
for p in prod_ok[:10]:   # primeros 10
    print(f'  {p.codigo:<10} {p.nombre:<32} {p.id_categoria:>4} '
          f'{p.precio_unitario:>8.2f} {p.stock_actual:>6} {p.stock_minimo:>5}')
print(f'  ... ({len(prod_ok) - 10} productos más)')


Productos aceptados : 52
Productos rechazados: 0

  CÓDIGO     NOMBRE                            CAT   PRECIO  STOCK   MÍN
  --------------------------------------------------------------------
  TORN-M4    Tornillo M4 x 20mm zinc             1     0.08   2000   500
  TORN-M6    Tornillo M6 x 30mm zinc             1     0.12   1500   400
  TORN-M8    Tornillo M8 x 40mm zinc             1     0.18    900   300
  TORN-M10   Tornillo M10 x 50mm zinc            1     0.25    600   200
  TURC-M6    Tuerca M6 hexagonal zinc            1     0.06   3000   800
  TURC-M8    Tuerca M8 hexagonal zinc            1     0.09   2500   700
  ROND-M6    Rondana plana M6 galv.              1     0.04   4000  1000
  ROND-M8    Rondana plana M8 galv.              1     0.05   3500   900
  CLAV-2P    Clavo 2" común caja 1kg             1     1.90     80    30
  CLAV-3P    Clavo 3" común caja 1kg             1     2.10     60    20
  ... (42 productos más)


### 2.3 Importación de movimientos


In [6]:
# Los movimientos necesitan el conjunto de códigos válidos
codigos_validos = {p.codigo for p in prod_ok}

mov_ok, mov_err = importar_movimientos(
    'data/entrada/movimientos.csv', codigos_validos, ids_existentes=set()
)

entradas = [m for m in mov_ok if m.tipo == 'E']
salidas  = [m for m in mov_ok if m.tipo == 'S']

print(f'Movimientos aceptados : {len(mov_ok)}')
print(f'  Entradas (E)        : {len(entradas)}')
print(f'  Salidas  (S)        : {len(salidas)}')
print(f'Movimientos rechazados: {len(mov_err)}')
print()
print('Primeros 5 movimientos:')
for m in mov_ok[:5]:
    tipo_txt = 'Entrada' if m.tipo == 'E' else 'Salida '
    print(f'  [{m.id_movimiento:>3}] {m.codigo_producto:<10} {tipo_txt}  '
          f'{m.cantidad:>5} uds  {m.fecha}')


Movimientos aceptados : 261
  Entradas (E)        : 87
  Salidas  (S)        : 174
Movimientos rechazados: 0

Primeros 5 movimientos:
  [  1] DEST-PL6   Entrada     55 uds  2025-01-05
  [  2] TORN-M4    Entrada     34 uds  2025-01-08
  [  3] TURC-M8    Entrada     74 uds  2025-01-09
  [  4] TABL-1P    Entrada     49 uds  2025-01-09
  [  5] TALDR-2    Entrada     53 uds  2025-01-10


### 2.4 Validación de datos con errores

El importador no detiene el proceso al encontrar una fila inválida:
la acumula en la lista de rechazadas con su motivo y continúa.


In [7]:
prod2_ok, prod2_err = importar_productos('data/entrada/productos_con_errores.csv')

print(f'Aceptados: {len(prod2_ok)}  |  Rechazados: {len(prod2_err)}')
print()
print('Filas rechazadas:')
for r in prod2_err:
    print(f'  Fila {r["fila"]:>2}: {r["motivo"]}')
print()
print('Filas aceptadas del archivo con errores:')
for p in prod2_ok:
    print(f'  {p.codigo}  {p.nombre}')


Aceptados: 2  |  Rechazados: 6

Filas rechazadas:
  Fila  3: 'codigo' no puede estar vacío
  Fila  4: 'nombre' no puede estar vacío
  Fila  5: 'precio_unitario' no puede ser negativo: -2.0
  Fila  6: 'stock_actual' no puede ser negativo: -3
  Fila  7: 'id_categoria' no es un entero: 'X'
  Fila  9: 'codigo' excede 10 caracteres: 'CODIGOMUYLARGO'

Filas aceptadas del archivo con errores:
  PROD-V1  Válido normal
  PROD-V6  Válido también


## 3. Módulo de Almacenamiento (`src/almacenamiento.py`)

Persiste los objetos validados en archivos binarios de longitud fija.
Implementa la lógica de **upsert** para productos y **append** para movimientos.


### 3.1 Guardar los datos importados


In [8]:
import src.almacenamiento as alm

# Categorías (escritura completa)
alm.guardar_categorias(cats_ok)

# Productos (upsert: inserta si no existe, actualiza stock si ya existe)
res_prod = alm.guardar_productos(prod_ok)

# Movimientos (append: agrega al final sin sobreescribir)
alm.registrar_importacion('categorias.csv')
alm.registrar_importacion('productos.csv')
n_mov = alm.guardar_movimientos(mov_ok)
alm.registrar_importacion('movimientos.csv')

print('Resultado de guardar_productos:')
print(f'  Insertados : {res_prod["insertados"]}')
print(f'  Actualizados: {res_prod["actualizados"]}')
print(f'  Movimientos escritos: {n_mov}')


Resultado de guardar_productos:
  Insertados : 52
  Actualizados: 0
  Movimientos escritos: 261


### 3.2 Verificación de archivos `.dat` generados


In [9]:
from src.modelos import TAMANO_PRODUCTO, TAMANO_CATEGORIA, TAMANO_MOVIMIENTO

archivos = {
    'categorias.dat'  : TAMANO_CATEGORIA,
    'productos.dat'   : TAMANO_PRODUCTO,
    'movimientos.dat' : TAMANO_MOVIMIENTO,
    'importaciones.dat': 60,
}

print(f'  {"Archivo":<22} {"Tamaño":>10} {"Registros":>10}')
print('  ' + '-'*44)
for nombre, tamano_reg in archivos.items():
    ruta = os.path.join('data', 'binarios', nombre)
    tam  = os.path.getsize(ruta)
    regs = tam // tamano_reg
    print(f'  {nombre:<22} {tam:>8} B  {regs:>8} regs')


  Archivo                    Tamaño  Registros
  --------------------------------------------
  categorias.dat              204 B         6 regs
  productos.dat              3952 B        52 regs
  movimientos.dat           18009 B       261 regs
  importaciones.dat           180 B         3 regs


### 3.3 Prueba de upsert – reimportar productos

Al importar el mismo archivo de productos por segunda vez, el sistema **actualiza**
el `stock_actual` de los existentes en lugar de crear registros duplicados.
El total de registros en disco debe permanecer igual.


In [10]:
productos_antes = alm.leer_productos()
n_antes = len(productos_antes)

# Simulamos una segunda importación con stocks modificados
prod_v2, _ = importar_productos('data/entrada/productos.csv')
# Modificamos el stock del primero para simular cambio real
prod_v2[0].stock_actual = 9999
res2 = alm.guardar_productos(prod_v2)

productos_despues = alm.leer_productos()
n_despues = len(productos_despues)

producto_actualizado = next(p for p in productos_despues if p.codigo == prod_v2[0].codigo)

print(f'Registros antes  : {n_antes}')
print(f'Registros después: {n_despues}  ← mismo número (no se duplican)')
print(f'Resultado upsert : {res2}')
print(f'Stock de {producto_actualizado.codigo}: {producto_actualizado.stock_actual}  ← actualizado a 9999')

assert n_antes == n_despues, 'ERROR: se duplicaron registros'
assert producto_actualizado.stock_actual == 9999
print('\nAserciones OK – el upsert funciona correctamente.')

# Restaurar el stock original
prod_v2[0].stock_actual = prod_ok[0].stock_actual
alm.guardar_productos(prod_v2)


Registros antes  : 52
Registros después: 52  ← mismo número (no se duplican)
Resultado upsert : {'insertados': 0, 'actualizados': 52}
Stock de TORN-M4: 9999  ← actualizado a 9999

Aserciones OK – el upsert funciona correctamente.


{'insertados': 0, 'actualizados': 52}

### 3.4 Prueba de no duplicación – reimportar movimientos

Si se intenta importar un archivo de movimientos ya procesado, todos los registros
son rechazados porque sus `id_movimiento` ya existen en el binario.


In [11]:
ids_existentes = alm.obtener_ids_movimientos()
mov2_ok, mov2_err = importar_movimientos(
    'data/entrada/movimientos.csv', codigos_validos, ids_existentes
)

print(f'Segunda importación del mismo archivo:')
print(f'  Aceptados : {len(mov2_ok)}  ← cero, todos ya existen')
print(f'  Rechazados: {len(mov2_err)}')
print(f'  Motivo de los primeros 3 rechazos:')
for r in mov2_err[:3]:
    print(f'    Fila {r["fila"]}: {r["motivo"]}')

assert len(mov2_ok) == 0, 'ERROR: se aceptaron duplicados'
print('\nAserción OK – no se duplican movimientos.')


Segunda importación del mismo archivo:
  Aceptados : 0  ← cero, todos ya existen
  Rechazados: 261
  Motivo de los primeros 3 rechazos:
    Fila 2: id_movimiento duplicado: 1
    Fila 3: id_movimiento duplicado: 2
    Fila 4: id_movimiento duplicado: 3

Aserción OK – no se duplican movimientos.


### 3.5 Registro de archivos importados

`importaciones.dat` guarda el nombre de cada archivo ya procesado,
permitiendo al menú advertir al usuario antes de reimportar.


In [12]:
print('Archivos registrados en importaciones.dat:')
from src.almacenamiento import _leer_nombres_importados, RUTA_IMPORTACIONES
for nombre in _leer_nombres_importados(RUTA_IMPORTACIONES):
    print(f'  ✓ {nombre}')
print()
print(f'"categorias.csv" ya importado: {alm.archivo_ya_importado("categorias.csv")}')
print(f'"nuevo_lote.csv" ya importado: {alm.archivo_ya_importado("nuevo_lote.csv")}')


Archivos registrados en importaciones.dat:
  ✓ categorias.csv
  ✓ productos.csv
  ✓ movimientos.csv

"categorias.csv" ya importado: True
"nuevo_lote.csv" ya importado: False


## 4. Consultas


### 4.1 Inventario actual

Muestra todos los productos almacenados. Los marcados con `!` tienen
`stock_actual` por debajo de `stock_minimo`.


In [13]:
productos  = alm.leer_productos()
categorias = alm.leer_categorias()
nombres_cat = {c.id: c.nombre for c in categorias}

print(f'Total de productos en inventario: {len(productos)}')
print()
print(f'  {"CÓDIGO":<10} {"NOMBRE":<32} {"CATEGORÍA":<22} {"PRECIO":>8} {"STOCK":>6} {"MÍN":>5}')
print('  ' + '-'*84)
bajo_stock = 0
for p in sorted(productos, key=lambda x: x.codigo):
    cat = nombres_cat.get(p.id_categoria, '?')
    alerta = ' !' if p.stock_actual < p.stock_minimo else ''
    if alerta:
        bajo_stock += 1
    print(f'  {p.codigo:<10} {p.nombre:<32} {cat:<22} '
          f'{p.precio_unitario:>8.2f} {p.stock_actual:>6}{alerta}')
print('  ' + '-'*84)
print(f'  Productos con stock bajo (!): {bajo_stock}')


Total de productos en inventario: 52

  CÓDIGO     NOMBRE                           CATEGORÍA                PRECIO  STOCK   MÍN
  ------------------------------------------------------------------------------------
  BROD-2P    Brocha cerda 2"                  Pintura y Acabados         3.50     60
  BROD-4P    Brocha cerda 4"                  Pintura y Acabados         5.80     45
  CABL-12    Cable THW cal.12 rollo 50m       Electricidad              38.00     18
  CABL-14    Cable THW cal.14 rollo 50m       Electricidad              28.00     22
  CALAD-1    Caladora 500W                    Herramientas Eléctricas    78.00      1 !
  CINT-5M    Cinta métrica 5 m                Herramientas Manuales      6.90     30
  CLAV-2P    Clavo 2" común caja 1kg          Tornillería                1.90     80
  CLAV-3P    Clavo 3" común caja 1kg          Tornillería                2.10     60
  CODE-1P    Codo PVC 1" 90°                  Plomería                   0.45    300
  CODE-2P    Cod

### 4.2 Estadísticas básicas del inventario

Resumen numérico de los productos visibles: precio promedio, stock total y valor total en inventario.
Las mismas cifras aparecen al pie de la consulta de inventario en el menú de consola.


In [14]:
# Estadísticas sobre todos los productos cargados
precio_promedio = sum(p.precio_unitario for p in productos) / len(productos)
stock_total     = sum(p.stock_actual for p in productos)
valor_total     = sum(p.stock_actual * p.precio_unitario for p in productos)
bajo_stock      = sum(1 for p in productos if p.stock_actual < p.stock_minimo)

print(f'Productos en inventario : {len(productos)}')
print(f'Precio promedio         : L{precio_promedio:.2f}')
print(f'Stock total (uds)       : {stock_total:,}')
print(f'Valor total en inventario: L{valor_total:,.2f}')
print(f'Productos con stock bajo : {bajo_stock}')
print()

# Mismas estadísticas filtradas por categoría (ejemplo: cat 1 = Tornillería)
CAT_ID = 1
cat_nombre = next((c.nombre for c in categorias if c.id == CAT_ID), f'Cat.{CAT_ID}')
subset = [p for p in productos if p.id_categoria == CAT_ID]
if subset:
    pp = sum(p.precio_unitario for p in subset) / len(subset)
    st = sum(p.stock_actual for p in subset)
    vt = sum(p.stock_actual * p.precio_unitario for p in subset)
    print(f'[Filtro: {cat_nombre}]')
    print(f'  Productos            : {len(subset)}')
    print(f'  Precio promedio      : L{pp:.2f}')
    print(f'  Stock total          : {st:,}')
    print(f'  Valor total          : L{vt:,.2f}')


Productos en inventario : 52
Precio promedio         : L21.49
Stock total (uds)       : 20,431
Valor total en inventario: L16,266.90
Productos con stock bajo : 6

[Filtro: Tornillería]
  Productos            : 10
  Precio promedio      : L0.49
  Stock total          : 18,140
  Valor total          : L1,670.00


### 4.2 Movimientos por período

Filtra los movimientos entre dos fechas y muestra el balance de entradas/salidas.


In [15]:
movimientos = alm.leer_movimientos()

DESDE = '2026-01-01'
HASTA = '2026-06-14'

periodo = [m for m in movimientos if DESDE <= m.fecha <= HASTA]
entradas_p = sum(m.cantidad for m in periodo if m.tipo == 'E')
salidas_p  = sum(m.cantidad for m in periodo if m.tipo == 'S')

print(f'Movimientos entre {DESDE} y {HASTA}: {len(periodo)}')
print(f'  Unidades entrada : {entradas_p}')
print(f'  Unidades salida  : {salidas_p}')
print(f'  Balance neto     : {entradas_p - salidas_p:+d}')
print()
print(f'  {"ID":>5} {"PRODUCTO":<10} {"TIPO":<8} {"CANT":>6} {"FECHA"}')
print('  ' + '-'*42)
for m in sorted(periodo, key=lambda x: x.fecha)[:15]:
    tipo_txt = 'Entrada' if m.tipo == 'E' else 'Salida '
    print(f'  {m.id_movimiento:>5} {m.codigo_producto:<10} {tipo_txt:<8} '
          f'{m.cantidad:>6} {m.fecha}')
if len(periodo) > 15:
    print(f'  ... ({len(periodo) - 15} movimientos más en el período)')


Movimientos entre 2026-01-01 y 2026-06-14: 78
  Unidades entrada : 1937
  Unidades salida  : 1639
  Balance neto     : +298

     ID PRODUCTO   TIPO       CANT FECHA
  ------------------------------------------
    184 CALAD-1    Salida       19 2026-01-07
    185 ROND-M8    Salida       26 2026-01-10
    186 SIER-C1    Salida       59 2026-01-19
    187 TURC-M6    Salida       52 2026-01-22
    188 TOMA-DBL   Salida       14 2026-01-28
    189 DEST-PH2   Salida       42 2026-01-30
    190 DEST-PL6   Salida       52 2026-02-01
    191 LLAV-1P    Salida       16 2026-02-06
    192 SIER-HK    Salida       60 2026-02-13
    193 TUBO-2P    Salida       10 2026-02-18
    194 MART-2A    Salida       44 2026-02-20
    195 CINT-5M    Salida       24 2026-02-20
    196 ROND-M8    Entrada     144 2026-02-21
    197 ESMER-1    Salida       38 2026-02-24
    198 PINT-BL    Salida       15 2026-03-02
  ... (63 movimientos más en el período)


## 5. Reportes (`src/reportes.py`)

Las funciones de reportes reciben listas del modelo y devuelven datos estructurados.
No hacen I/O: la responsabilidad de cargar y mostrar queda en el módulo que las llama.


### 5.1 Stock total por categoría

Suma el `stock_actual` y el valor (`stock × precio`) de todos los productos
agrupados por categoría.


In [16]:
from src.reportes import (
    stock_por_categoria, top_inmovilizado,
    ventas_mensuales_por_categoria, alertas_stock_bajo
)

reporte1 = stock_por_categoria(productos, categorias)

total_stock = sum(r['stock_total'] for r in reporte1)
total_valor = sum(r['valor_total'] for r in reporte1)

print(f'  {"CATEGORÍA":<25} {"PRODS":>6} {"STOCK":>8} {"VALOR (LPS)":>12}')
print('  ' + '-'*54)
for r in reporte1:
    print(f'  {r["nombre"]:<25} {r["num_productos"]:>6} {r["stock_total"]:>8} {r["valor_total"]:>12.2f}')
print('  ' + '-'*54)
print(f'  {"TOTAL":<25} {len(productos):>6} {total_stock:>8} {total_valor:>12.2f}')


  CATEGORÍA                  PRODS    STOCK  VALOR (LPS)
  ------------------------------------------------------
  Tornillería                   10    18140      1670.00
  Plomería                       7      962      1511.00
  Electricidad                   9      768      3659.00
  Pintura y Acabados             8      318      3727.20
  Herramientas Manuales         10      197      1640.70
  Herramientas Eléctricas        8       46      4059.00
  ------------------------------------------------------
  TOTAL                         52    20431     16266.90


### 5.2 Top 10 productos con stock inmovilizado

Identifica productos **sin movimientos de salida en los últimos N días**.
El valor inmovilizado es `stock_actual × precio_unitario` (dinero bloqueado en inventario).


In [17]:
DIAS_VENTANA = 90
TOP_N = 10

reporte2 = top_inmovilizado(productos, movimientos, n=TOP_N, dias=DIAS_VENTANA)

print(f'Top {TOP_N} productos sin salidas en los últimos {DIAS_VENTANA} días:')
print()
print(f'  {"#":<3} {"CÓDIGO":<10} {"NOMBRE":<30} {"STOCK":>6} '
      f'{"VALOR (LPS)":>10} {"ÚLT. SALIDA":<14}')
print('  ' + '-'*76)
for i, r in enumerate(reporte2, 1):
    salida = r['ultima_salida'] or 'Sin salidas'
    print(f'  {i:<3} {r["codigo"]:<10} {r["nombre"]:<30} {r["stock_actual"]:>6} '
          f'{r["valor_inmovilizado"]:>10.2f} {salida:<14}')
print()
total_inm = sum(r['valor_inmovilizado'] for r in reporte2)
print(f'  Valor total inmovilizado (top {TOP_N}): L{total_inm:,.2f}')


Top 10 productos sin salidas en los últimos 90 días:

  #   CÓDIGO     NOMBRE                          STOCK VALOR (LPS) ÚLT. SALIDA   
  ----------------------------------------------------------------------------
  1   CABL-12    Cable THW cal.12 rollo 50m         18     684.00 2025-07-04    
  2   PINT-GR    Pintura látex gris 4L              35     647.50 2025-06-01    
  3   CABL-14    Cable THW cal.14 rollo 50m         22     616.00 2025-08-18    
  4   FOCOS-L9   Foco LED 9W E27                   200     580.00 2026-03-08    
  5   ESMA-BL    Esmalte blanco brillante 1L        30     336.00 2025-09-19    
  6   TUBO-2P    Tubo PVC 2" x 6m                   30     294.00 2026-02-18    
  7   ESMA-NG    Esmalte negro mate 1L              25     280.00 2025-10-10    
  8   INTE-SNC   Interruptor sencillo              150     270.00 2025-10-08    
  9   BROD-4P    Brocha cerda 4"                    45     261.00 2025-08-16    
  10  ROLLO-1    Rodillo felpa 9" + bandeja         28  

### 5.3 Ventas mensuales por categoría

Totaliza las unidades **salidas** (tipo `S`) agrupadas por mes y categoría.
Permite identificar tendencias de demanda y estacionalidad.


In [18]:
reporte3 = ventas_mensuales_por_categoria(movimientos, productos, categorias)

print(f'  {"MES":<10} {"CATEGORÍA":<25} {"UNIDADES":>9}')
print('  ' + '-'*46)

mes_actual = None
for r in reporte3:
    mes_str = f'{r["anio"]}-{r["mes"]:02d}'
    if mes_str != mes_actual:
        if mes_actual is not None:
            print()
        mes_actual = mes_str
    print(f'  {mes_str:<10} {r["nombre_categoria"]:<25} {r["unidades"]:>9}')

print()
total_uds = sum(r['unidades'] for r in reporte3)
print(f'  Total de unidades vendidas en el período: {total_uds}')
print(f'  Meses con actividad: {len(set((r["anio"],r["mes"]) for r in reporte3))}')


  MES        CATEGORÍA                  UNIDADES
  ----------------------------------------------
  2025-04    Electricidad                      3
  2025-04    Herramientas Eléctricas          71
  2025-04    Herramientas Manuales           120
  2025-04    Pintura y Acabados                6
  2025-04    Tornillería                       8

  2025-05    Electricidad                      3
  2025-05    Herramientas Eléctricas          88
  2025-05    Herramientas Manuales            99
  2025-05    Pintura y Acabados               68
  2025-05    Tornillería                     135

  2025-06    Herramientas Eléctricas          58
  2025-06    Herramientas Manuales            64
  2025-06    Pintura y Acabados               42
  2025-06    Tornillería                      54

  2025-07    Electricidad                      2
  2025-07    Herramientas Eléctricas           9
  2025-07    Herramientas Manuales           150
  2025-07    Pintura y Acabados               26
  2025-07    Torn

## 6. Alertas de Stock Bajo

Lista los productos cuyo `stock_actual` está por debajo del mínimo establecido.
Ordenados de mayor a menor déficit (el más crítico primero).


### 6.1 Usando el mínimo de cada producto


In [19]:
alertas = alertas_stock_bajo(productos)

print(f'Productos con stock bajo su propio mínimo: {len(alertas)}')
print()
print(f'  {"CÓDIGO":<10} {"NOMBRE":<30} {"ACTUAL":>7} {"MÍNIMO":>7} {"DÉFICIT":>8}')
print('  ' + '-'*62)
for a in alertas:
    cat = nombres_cat.get(a['id_categoria'], '?')
    print(f'  {a["codigo"]:<10} {a["nombre"]:<30} '
          f'{a["stock_actual"]:>7} {a["minimo"]:>7} {a["diferencia"]:>8}')


Productos con stock bajo su propio mínimo: 6

  CÓDIGO     NOMBRE                          ACTUAL  MÍNIMO  DÉFICIT
  --------------------------------------------------------------
  NIVE-60    Nivel de burbuja 60 cm               3      10       -7
  MART-2A    Martillo peña 2 lb                   2       8       -6
  SIER-HK    Serrucho hoja fina 22"               2       8       -6
  LIJAD-1    Lijadora orbital 1/4 hoja            1       5       -4
  TABL-2P    Tablero 1 fase 8 circuitos           1       5       -4
  CALAD-1    Caladora 500W                        1       4       -3


### 6.2 Con umbral global configurable

El umbral global es útil para lanzar una revisión amplia sin considerar
los mínimos individuales, por ejemplo durante un inventario físico.


In [20]:
UMBRAL = 15
alertas_u = alertas_stock_bajo(productos, umbral=UMBRAL)

print(f'Productos con stock < {UMBRAL} unidades (umbral global): {len(alertas_u)}')
print()
print(f'  {"CÓDIGO":<10} {"NOMBRE":<30} {"ACTUAL":>7} {"MÍNIMO":>7} {"DÉFICIT":>8}')
print('  ' + '-'*62)
for a in alertas_u:
    print(f'  {a["codigo"]:<10} {a["nombre"]:<30} '
          f'{a["stock_actual"]:>7} {a["minimo"]:>7} {a["diferencia"]:>8}')


Productos con stock < 15 unidades (umbral global): 13

  CÓDIGO     NOMBRE                          ACTUAL  MÍNIMO  DÉFICIT
  --------------------------------------------------------------
  LIJAD-1    Lijadora orbital 1/4 hoja            1      15      -14
  CALAD-1    Caladora 500W                        1      15      -14
  TABL-2P    Tablero 1 fase 8 circuitos           1      15      -14
  MART-2A    Martillo peña 2 lb                   2      15      -13
  SIER-HK    Serrucho hoja fina 22"               2      15      -13
  NIVE-60    Nivel de burbuja 60 cm               3      15      -12
  ESMER-2    Esmeriladora angular 7"              5      15      -10
  TALDR-2    Taladro inalámbrico 20V              6      15       -9
  SOPLA-1    Sopladora/aspiradora 600W            6      15       -9
  SIER-C1    Sierra circular 7.25"                7      15       -8
  TABL-1P    Tablero 1 fase 4 circuitos           7      15       -8
  TALDR-1    Taladro percutor 650W                9 

## 7. Pruebas de Consistencia

Verificaciones formales para las pruebas mínimas requeridas en la Fase I.


### 7.1 Consistencia del número de registros importados


In [21]:
prod_csv, _  = importar_productos('data/entrada/productos.csv')
cats_csv, _  = importar_categorias('data/entrada/categorias.csv')
mov_csv, _   = importar_movimientos('data/entrada/movimientos.csv',
                                     codigos_validos, ids_existentes=set())

prod_bin = alm.leer_productos()
cats_bin = alm.leer_categorias()
mov_bin  = alm.leer_movimientos()

print('Consistencia CSV → binario:')
print(f'  Categorías  CSV={len(cats_csv)}  DAT={len(cats_bin)}  OK={len(cats_csv)==len(cats_bin)}')
print(f'  Productos   CSV={len(prod_csv)}  DAT={len(prod_bin)}  OK={len(prod_csv)==len(prod_bin)}')
print(f'  Movimientos CSV={len(mov_csv)}  DAT={len(mov_bin)}  OK={len(mov_csv)==len(mov_bin)}')

assert len(cats_csv) == len(cats_bin)
assert len(prod_csv) == len(prod_bin)
assert len(mov_csv)  == len(mov_bin)
print('\nTodas las aserciones de consistencia pasaron.')


Consistencia CSV → binario:
  Categorías  CSV=6  DAT=6  OK=True
  Productos   CSV=52  DAT=52  OK=True
  Movimientos CSV=261  DAT=261  OK=True

Todas las aserciones de consistencia pasaron.


### 7.2 No duplicación al reimportar el mismo CSV


In [22]:
# Intentar reimportar movimientos con IDs ya existentes
ids_ya_en_disco = alm.obtener_ids_movimientos()
mov_dup_ok, mov_dup_err = importar_movimientos(
    'data/entrada/movimientos.csv', codigos_validos, ids_ya_en_disco
)
alm.guardar_movimientos(mov_dup_ok)   # debería agregar 0 registros

mov_bin_v2 = alm.leer_movimientos()

print(f'Movimientos aceptados en reimportación: {len(mov_dup_ok)}')
print(f'Movimientos rechazados (duplicados)   : {len(mov_dup_err)}')
print(f'Total en disco antes: {len(mov_bin)}')
print(f'Total en disco ahora: {len(mov_bin_v2)}')

assert len(mov_dup_ok) == 0, 'Se aceptaron duplicados'
assert len(mov_bin) == len(mov_bin_v2), 'Cambió el total de movimientos'
print('\nAserción OK – reimportar no genera duplicados.')


Movimientos aceptados en reimportación: 0
Movimientos rechazados (duplicados)   : 261
Total en disco antes: 261
Total en disco ahora: 261

Aserción OK – reimportar no genera duplicados.


### 7.3 Correctitud de los reportes


In [23]:
# Stock por categoría: la suma total debe coincidir con la suma individual
rep1 = stock_por_categoria(prod_bin, cats_bin)
suma_rep   = sum(r['stock_total'] for r in rep1)
suma_dir   = sum(p.stock_actual for p in prod_bin)
assert suma_rep == suma_dir, f'{suma_rep} != {suma_dir}'
print(f'Stock total por categoría = stock total individual = {suma_rep} ✓')

# Alertas: todos deben tener stock_actual < minimo
alertas_v = alertas_stock_bajo(prod_bin)
assert all(a['stock_actual'] < a['minimo'] for a in alertas_v)
print(f'Todas las alertas cumplen stock_actual < mínimo ({len(alertas_v)} productos) ✓')

# Ventas mensuales: solo deben incluir movimientos tipo S
rep3 = ventas_mensuales_por_categoria(mov_bin, prod_bin, cats_bin)
total_ventas_rep  = sum(r['unidades'] for r in rep3)
total_salidas_dir = sum(m.cantidad for m in mov_bin if m.tipo == 'S')
assert total_ventas_rep == total_salidas_dir, f'{total_ventas_rep} != {total_salidas_dir}'
print(f'Total unidades ventas = total salidas directas = {total_ventas_rep} ✓')

# Top inmovilizado: valor debe ser stock × precio
rep2 = top_inmovilizado(prod_bin, mov_bin)
for r in rep2:
    p_match = next(p for p in prod_bin if p.codigo == r['codigo'])
    esperado = round(p_match.stock_actual * p_match.precio_unitario, 2)
    assert r['valor_inmovilizado'] == esperado, f"{r['codigo']}: {r['valor_inmovilizado']} != {esperado}"
print(f'Valor inmovilizado = stock × precio para todos los {len(rep2)} registros ✓')

print('\nTodas las pruebas de correctitud pasaron.')


Stock total por categoría = stock total individual = 20431 ✓
Todas las alertas cumplen stock_actual < mínimo (6 productos) ✓
Total unidades ventas = total salidas directas = 3876 ✓
Valor inmovilizado = stock × precio para todos los 10 registros ✓

Todas las pruebas de correctitud pasaron.


## 8. Resumen de Pruebas


In [24]:
print('=' * 55)
print('  RESUMEN – FerroAnalytics Fase I')
print('=' * 55)

pruebas = [
    ('Serialización round-trip (3 modelos)',        True),
    ('Importación categorías sin errores',          len(cats_err) == 0),
    ('Importación productos sin errores',           len(prod_err) == 0),
    ('Importación movimientos sin errores',         len(mov_err) == 0),
    ('Validación archivo con errores (6 rechazo)', len(prod2_err) == 6),
    ('Upsert productos (sin duplicados)',           len(prod_bin) == len(prod_csv)),
    ('No duplicación al reimportar movimientos',    len(mov_dup_ok) == 0),
    ('Registro de importaciones (.dat)',            alm.archivo_ya_importado('categorias.csv')),
    ('Reporte 1: stock suma correcta',              suma_rep == suma_dir),
    ('Reporte 2: valor = stock × precio',           True),
    ('Reporte 3: ventas = total salidas',           total_ventas_rep == total_salidas_dir),
    ('Reporte 4: alertas con stock < mínimo',       all(a["stock_actual"] < a["minimo"] for a in alertas_v)),
]

for nombre, resultado in pruebas:
    estado = 'PASS' if resultado else 'FAIL'
    print(f'  [{estado}]  {nombre}')

n_pass = sum(1 for _, r in pruebas if r)
print()
print(f'  Resultado: {n_pass}/{len(pruebas)} pruebas pasadas')
print('=' * 55)


  RESUMEN – FerroAnalytics Fase I
  [PASS]  Serialización round-trip (3 modelos)
  [PASS]  Importación categorías sin errores
  [PASS]  Importación productos sin errores
  [PASS]  Importación movimientos sin errores
  [PASS]  Validación archivo con errores (6 rechazo)
  [PASS]  Upsert productos (sin duplicados)
  [PASS]  No duplicación al reimportar movimientos
  [PASS]  Registro de importaciones (.dat)
  [PASS]  Reporte 1: stock suma correcta
  [PASS]  Reporte 2: valor = stock × precio
  [PASS]  Reporte 3: ventas = total salidas
  [PASS]  Reporte 4: alertas con stock < mínimo

  Resultado: 12/12 pruebas pasadas
